In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report

# What are RNNs?

Traditional neural networks (including CNNs) assume all inputs are **independent**. But what about data where **order matters**?

## Examples of Sequential Data

- **Text:** "I love this movie" vs "I hate this movie"
- **Time series:** Stock prices over time
- **Audio:** Speech signals
- **Video:** Frames in sequence

> RNNs have **MEMORY**! They remember what came before.


## The Core Idea: Memory

```
Regular NN:  Input → Hidden → Output
             (No memory, each input independent)

RNN:         Input → Hidden → Output
                       ↑
                    Memory (hidden state)
                    Remembers previous inputs!
```


# RNN Unrolled Through Time

For a sequence of 3 words: **"I love AI"**


## Time Step 1: `x₁ = "I"`

```
h₁ = f(W*x₁ + U*h₀ + b)   ← h₀ is initial hidden state (zeros)
y₁ = g(V*h₁ + c)
```

## Time Step 2: `x₂ = "love"`

```
h₂ = f(W*x₂ + U*h₁ + b)   ← Uses h₁ (memory from step 1!)
y₂ = g(V*h₂ + c)
```

## Time Step 3: `x₃ = "AI"`

```
h₃ = f(W*x₃ + U*h₂ + b)   ← Uses h₂ (memory from step 2!)
y₃ = g(V*h₃ + c)
```


## Key Insight

- **Same weights** (W, U, V) used at every time step!
- Hidden state `h` carries information forward
- This is how RNNs **"remember"** previous inputs

In [2]:
print("\n" + "="*60)
print("BUILDING A SIMPLE RNN FROM SCRATCH")
print("="*60)

class SimpleRNN(nn.Module):
    """A simple RNN implementation"""
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.hidden_size = hidden_size
        
        # Weights
        self.W_xh = nn.Linear(input_size, hidden_size)   # Input to hidden
        self.W_hh = nn.Linear(hidden_size, hidden_size)  # Hidden to hidden
        self.W_hy = nn.Linear(hidden_size, output_size)  # Hidden to output
        
        self.tanh = nn.Tanh()
    
    def forward(self, x, hidden=None):
        """
        x: (batch_size, seq_length, input_size)
        hidden: (batch_size, hidden_size)
        """
        batch_size, seq_length, _ = x.size()
        
        if hidden is None:
            hidden = torch.zeros(batch_size, self.hidden_size).to(x.device)
        
        outputs = []
        
        # Process each time step
        for t in range(seq_length):
            x_t = x[:, t, :]  # (batch_size, input_size)
            
            # Update hidden state
            hidden = self.tanh(self.W_xh(x_t) + self.W_hh(hidden))
            
            # Compute output
            output = self.W_hy(hidden)
            outputs.append(output)
        
        # Stack outputs
        outputs = torch.stack(outputs, dim=1)  # (batch_size, seq_length, output_size)
        
        return outputs, hidden

# Test the RNN
print("Testing SimpleRNN...")
input_size = 10
hidden_size = 20
output_size = 1
seq_length = 5
batch_size = 3

rnn = SimpleRNN(input_size, hidden_size, output_size)
x = torch.randn(batch_size, seq_length, input_size)
outputs, hidden = rnn(x)

print(f"Input shape: {x.shape}")       # (3, 5, 10)
print(f"Output shape: {outputs.shape}") # (3, 5, 1)
print(f"Hidden shape: {hidden.shape}")  # (3, 20)


BUILDING A SIMPLE RNN FROM SCRATCH
Testing SimpleRNN...
Input shape: torch.Size([3, 5, 10])
Output shape: torch.Size([3, 5, 1])
Hidden shape: torch.Size([3, 20])


In [3]:
print("\n" + "="*60)
print("SENTIMENT ANALYSIS WITH RNN")
print("="*60)

print("""
Task: Classify movie reviews as positive or negative

Example:
  "This movie is great!" → Positive
  "This movie is terrible!" → Negative

Challenge: Words have order and context!
""")

# Create synthetic sentiment data
np.random.seed(42)

# Vocabulary
vocab = {
    '<pad>': 0, '<unk>': 1,
    'good': 2, 'great': 3, 'excellent': 4, 'amazing': 5, 'wonderful': 6,
    'bad': 7, 'terrible': 8, 'awful': 9, 'horrible': 10, 'poor': 11,
    'movie': 12, 'film': 13, 'acting': 14, 'story': 15, 'plot': 16,
    'the': 17, 'a': 18, 'is': 19, 'was': 20, 'and': 21, 'but': 22,
    'i': 23, 'loved': 24, 'hated': 25, 'it': 26, 'this': 27
}

# Reverse vocabulary
idx2word = {v: k for k, v in vocab.items()}

def generate_review(sentiment, length=10):
    """Generate a synthetic review"""
    if sentiment == 1:  # Positive
        positive_words = ['good', 'great', 'excellent', 'amazing', 'wonderful', 'loved']
        words = ['the', 'movie', 'was'] + [np.random.choice(positive_words) for _ in range(length-3)]
    else:  # Negative
        negative_words = ['bad', 'terrible', 'awful', 'horrible', 'poor', 'hated']
        words = ['the', 'movie', 'was'] + [np.random.choice(negative_words) for _ in range(length-3)]
    
    # Shuffle to make it harder
    np.random.shuffle(words)
    
    # Convert to indices
    indices = [vocab.get(w, 1) for w in words]
    return indices

# Generate dataset
n_samples = 1000
max_length = 10

X_data = []
y_data = []

for _ in range(n_samples // 2):
    # Positive reviews
    X_data.append(generate_review(1, max_length))
    y_data.append(1)
    
    # Negative reviews
    X_data.append(generate_review(0, max_length))
    y_data.append(0)

X_data = np.array(X_data)
y_data = np.array(y_data)

# Shuffle
indices = np.random.permutation(len(X_data))
X_data = X_data[indices]
y_data = y_data[indices]

# Split
split_idx = int(0.8 * len(X_data))
X_train, X_test = X_data[:split_idx], X_data[split_idx:]
y_train, y_test = y_data[:split_idx], y_data[split_idx:]

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Positive examples: {sum(y_train)}")
print(f"Negative examples: {len(y_train) - sum(y_train)}")


SENTIMENT ANALYSIS WITH RNN

Task: Classify movie reviews as positive or negative

Example:
  "This movie is great!" → Positive
  "This movie is terrible!" → Negative

Challenge: Words have order and context!

Training samples: 800
Test samples: 200
Positive examples: 406
Negative examples: 394


In [4]:
print("\n" + "="*60)
print("TRAINING SENTIMENT RNN")
print("="*60)

class SentimentRNN(nn.Module):
    """RNN for sentiment analysis"""
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(SentimentRNN, self).__init__()
        
        # Embedding layer: convert word indices to dense vectors
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # RNN layer
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        
        # Output layer
        self.fc = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # x: (batch_size, seq_length)
        embedded = self.embedding(x)  # (batch_size, seq_length, embedding_dim)
        
        # RNN forward
        output, hidden = self.rnn(embedded)
        # output: (batch_size, seq_length, hidden_size)
        # hidden: (1, batch_size, hidden_size)
        
        # Use last hidden state for classification
        hidden = hidden.squeeze(0)  # (batch_size, hidden_size)
        
        # Classify
        out = self.fc(hidden)
        out = self.sigmoid(out)
        
        return out

# Create model
vocab_size = len(vocab)
embedding_dim = 32
hidden_size = 64
output_size = 1

model = SentimentRNN(vocab_size, embedding_dim, hidden_size, output_size)
print("SentimentRNN Architecture:")
print(model)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Training
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Convert to tensors
X_train_tensor = torch.LongTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).reshape(-1, 1)
X_test_tensor = torch.LongTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).reshape(-1, 1)

# Training loop
epochs = 20
train_losses = []
test_accuracies = []

print("\nTraining RNN for sentiment analysis...")
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        test_outputs = model(X_test_tensor)
        test_preds = (test_outputs > 0.5).float()
        accuracy = (test_preds == y_test_tensor).float().mean().item() * 100
    
    test_accuracies.append(accuracy)
    
    if epoch % 4 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}, Test Acc = {accuracy:.2f}%")

print(f"\nFinal Test Accuracy: {test_accuracies[-1]:.2f}%")


TRAINING SENTIMENT RNN
SentimentRNN Architecture:
SentimentRNN(
  (embedding): Embedding(28, 32, padding_idx=0)
  (rnn): RNN(32, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Total parameters: 7,233

Training RNN for sentiment analysis...
Epoch 0: Loss = 0.6812, Test Acc = 61.00%
Epoch 4: Loss = 0.6088, Test Acc = 80.00%
Epoch 8: Loss = 0.5259, Test Acc = 98.00%
Epoch 12: Loss = 0.4023, Test Acc = 100.00%
Epoch 16: Loss = 0.2254, Test Acc = 100.00%

Final Test Accuracy: 100.00%


# Problem with Simple RNNs

- **Vanishing gradient:** Can't learn long-term dependencies
- If sequence is long, early information is lost

**Example:**
> "I grew up in France... [100 words] ... I speak fluent ___"
>
> → Simple RNN forgets "France" by the time it reaches the blank


# Solution: LSTM (Long Short-Term Memory)

- Has a **"cell state"** that can carry information long-term
- **Gates** control what to remember, forget, and output
- Three gates: **Forget, Input, Output**

In [7]:
class SentimentLSTM(nn.Module):
    """LSTM for sentiment analysis"""
    def __init__(self, vocab_size, embedding_dim, hidden_size, output_size):
        super(SentimentLSTM, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        hidden = hidden.squeeze(0)
        out = self.fc(hidden)
        out = self.sigmoid(out)
        return out

# Train LSTM
lstm_model = SentimentLSTM(vocab_size, embedding_dim, hidden_size, output_size)
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)

lstm_train_losses = []
lstm_test_accuracies = []

print("Training LSTM...")
for epoch in range(epochs):
    lstm_model.train()
    optimizer_lstm.zero_grad()
    outputs = lstm_model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer_lstm.step()
    
    lstm_train_losses.append(loss.item())
    
    lstm_model.eval()
    with torch.no_grad():
        test_outputs = lstm_model(X_test_tensor)
        test_preds = (test_outputs > 0.5).float()
        accuracy = (test_preds == y_test_tensor).float().mean().item() * 100
    
    lstm_test_accuracies.append(accuracy)
    
    if epoch % 4 == 0:
        print(f"Epoch {epoch}: Loss = {loss.item():.4f}, Test Acc = {accuracy:.2f}%")

print(f"\nFinal LSTM Test Accuracy: {lstm_test_accuracies[-1]:.2f}%")

Training LSTM...
Epoch 0: Loss = 0.7040, Test Acc = 53.00%
Epoch 4: Loss = 0.6543, Test Acc = 90.00%
Epoch 8: Loss = 0.6038, Test Acc = 97.00%
Epoch 12: Loss = 0.5443, Test Acc = 97.50%
Epoch 16: Loss = 0.4678, Test Acc = 99.50%

Final LSTM Test Accuracy: 100.00%
